# Match Cells
Use the saved global propensity model from `global_psm.ipynb` to predict a propensity score for each of the candidate treatment and control cells, and match each treatment cell to a set of control cells with similar propensity scores.

In [ ]:
# Select PA
site_id = 67967

In [ ]:
from pathlib import Path
import sys
import os
import ee
import geemap
import numpy as np
import pandas as pd
import geopandas as gpd
from sklearn.neighbors import NearestNeighbors

cur = Path.cwd().resolve()
for parent in [cur] + list(cur.parents):
    if parent.name == "tpae":
        os.chdir(parent)
        break

sys.path.insert(0, str((Path.cwd() / "src").resolve()))

from utils.variables import (
    PROJECT,
    COUNTRIES_ASSET_ID,
    EE_CRS_METERS,
    PSM_CELL_SIZE,
    TREATMENT_CELLS,
    CONTROL_CELLS,
    BIOME_ASSET_ID,
    CALIPER,
    N_NEIGHBORS,
    GLO30_ASSET_ID,
    ATC_ASSET_ID,
    POP_ASSET_ID,
    HGFC_ASSET_ID
)

from absolute_effectiveness.site_selector import SiteSelector
from psm.predict import load_propensity_artifacts, predict_propensity

ee.Authenticate()
ee.Initialize(project=PROJECT)

site_selector = SiteSelector()

EE_CRS_1km = ee.Projection(EE_CRS_METERS).atScale(PSM_CELL_SIZE)

## Data Prep
Load candidate cells and covariates.

In [ ]:
# Load valid grid cells for a given PA

PA_ID = str(site_id)

test_sites = site_selector.get_test_sites()
site_geom = site_selector.get_site_geom(test_sites, site_id)

# Import treatment and control cells and filter to PA
treatment_cells = gpd.read_parquet(TREATMENT_CELLS).to_crs(epsg=4326)
treatment_cells = treatment_cells[treatment_cells["WDPAID"] == PA_ID]
control_cells = gpd.read_parquet(CONTROL_CELLS).to_crs(epsg=4326)
control_cells = control_cells[control_cells["WDPAID"] == PA_ID]

print(f"Number of candidate treatment cells: {len(treatment_cells)}")
print(f"Number of candidate control cells: {len(control_cells)}")


# Convert to ee.FeatureCollection
treatment_fc = geemap.geopandas_to_ee(treatment_cells)
control_fc = geemap.geopandas_to_ee(control_cells)
all_cells = ee.FeatureCollection([treatment_fc, control_fc]).flatten()

# Add a unique cell_ID to each grid cell
cell_IDs = ee.List.sequence(0, all_cells.size().getInfo() - 1)
featureList = all_cells.toList(all_cells.size())
grid_fc = ee.FeatureCollection(
    cell_IDs.map(
        lambda cell_ID: ee.Feature(featureList.get(cell_ID)).set(
            {"cell_ID": cell_ID, "label": None}
        )
    )
)

In [ ]:
# Load covariates

elevation_ic = ee.ImageCollection(GLO30_ASSET_ID).select("DEM")
elevation = (
    elevation_ic
    .mosaic()
    .setDefaultProjection(elevation_ic.first().projection())
    .rename("elevation")
)
slope = ee.Terrain.slope(elevation).rename("slope")
treecover2000 = ee.Image(HGFC_ASSET_ID).select("treecover2000")
travel_time = (
    ee.Image(ATC_ASSET_ID)
    .select("accessibility").rename("travel_time")
)
log_pop_density = (
    ee.Image(POP_ASSET_ID)
    .select("population_count")
    .add(1) # handles zeros for log transform
    .log()
    .rename("log_pop_density")
)

# Resample covariates to 1km resolution

def resample(img):
    return (
        img.setDefaultProjection(EE_CRS_1km)
        .reduceResolution(reducer=ee.Reducer.mean(), maxPixels=4096)
        .reproject(EE_CRS_1km)
    )

elevation = resample(elevation)
slope = resample(slope)
treecover2000 = resample(treecover2000)
travel_time = resample(travel_time)
log_pop_density = resample(log_pop_density)

covariates = (
    elevation
    .addBands(slope)
    .addBands(treecover2000)
    .addBands(travel_time)
    .addBands(log_pop_density)
)

## Predict Propensity Scores
Calculate covariate values within cells and apply propensity model to predict propensity scores for each cell.

In [ ]:
# Aggregate covariates within grid cells
# will need to use mode reducer for categorical covariates

grid_fc = (
    covariates
    .reduceRegions(
        collection=grid_fc,
        reducer=ee.Reducer.mean(),
        scale=PSM_CELL_SIZE,
        crs=EE_CRS_1km,
    )
    .select("cell_ID", "elevation", "slope", "treecover2000", "travel_time", "log_pop_density", "protected")
)

# Convert cells to centroids
centroids = grid_fc.map(lambda cell: ee.Feature(cell).centroid())

# Assign country and ecoregion to each centroid

countries = ee.FeatureCollection(COUNTRIES_ASSET_ID)
ecoregions = ee.FeatureCollection(BIOME_ASSET_ID)

spatial_filter = ee.Filter.intersects(
    leftField='.geo',
    rightField='.geo',
    maxError=1
)

centroids = ee.Join.saveFirst('_match').apply(
    primary=centroids,
    secondary=countries,
    condition=spatial_filter
).map(lambda f: f
    .set('country', ee.Feature(f.get('_match')).get('country_na'))
    .set('_match', None)
)

centroids = ee.Join.saveFirst('_match').apply(
    primary=centroids,
    secondary=ecoregions,
    condition=spatial_filter
).map(lambda f: f
    .set('ecoregion', ee.Feature(f.get('_match')).get('ECO_ID'))
    .set('biome', ee.Feature(f.get('_match')).get('BIOME_NUM'))
    .set('_match', None)
)

# Convert centroids to dataframe
cells_list = centroids.getInfo()["features"]
cells_df = pd.DataFrame([feature["properties"] for feature in cells_list])
print(cells_df.head())

# Drop cells with any missing covariate or categorical attribute

required_cols = ["elevation", "slope", "treecover2000", "travel_time", 
                 "log_pop_density", "country", "ecoregion", "biome"]

n_before = len(cells_df)
n_missing_by_col = cells_df[required_cols].isna().sum()
cells_df = cells_df.dropna(subset=required_cols).reset_index(drop=True)
n_dropped = n_before - len(cells_df)

if n_dropped > 0:
    print(f"⚠ Dropped {n_dropped}/{n_before} cells ({n_dropped/n_before:.1%}) with missing values:")
    for col, n in n_missing_by_col.items():
        if n > 0:
            print(f"    {col}: {n}")

if n_dropped == 0:
    print("No cells dropped.")

In [ ]:
# Load the saved propensity score model from global_psm.ipynb

model_files = sorted(Path("models").glob("propensity_model_*.pkl"))
artifacts = load_propensity_artifacts(model_files[-1])
print(f"Loaded {model_files[-1].name}")
print(f"Training AUC: {artifacts['training_metadata']['auc']:.4f}")

# Predict propensity scores for each cell

cells_df["propensity_score"] = predict_propensity(cells_df, artifacts)

print(f"\nCells: {len(cells_df)}")
print(f"Treatment (protected=1): {(cells_df['protected'] == 1).sum()}")
print(f"Control (protected=0): {(cells_df['protected'] == 0).sum()}")
print(f"\nPropensity score distribution:")
print(cells_df["propensity_score"].describe())
print(f"\nFirst 5 rows:")
print(cells_df[["cell_ID", "protected", "biome", "country", "ecoregion", "propensity_score"]].head())

## Match Cells
Match each treatment cell to a set of control cells. Re-use of control cells is ok. Matching is based on similarity of propensity score, with exact-matching required for country and ecoregion. Ecoregion constraint relaxes to biome if no within-ecoregion controls exist.

In [ ]:
# Split treatment and control
treat_df = cells_df[cells_df["protected"] == 1].copy().reset_index(drop=True)
control_df = cells_df[cells_df["protected"] == 0].copy().reset_index(drop=True)

matches = []

# Implement exact-matching for country and ecoregion.

for (country, ecoregion), treat_sub in treat_df.groupby(["country", "ecoregion"]):
    control_country = control_df[control_df["country"] == country]

    if len(control_country) == 0:
        print(f"  ({country}, ecoregion {ecoregion}): no controls in country, "
              f"skipping {len(treat_sub)} treatment cells")
        continue

    control_sub = control_country[control_country["ecoregion"] == ecoregion]

    if len(control_sub) == 0:
        # No same-ecoregion controls in this country; fall back to same-biome
        biome = treat_sub["biome"].iloc[0]
        control_sub = control_country[control_country["biome"] == biome]
        fallback = "biome"
        print(f"  ({country}, ecoregion {ecoregion}): no within-ecoregion controls, "
              f"falling back to biome {biome} ({len(control_sub)} controls)")
    else:
        fallback = None

    if len(control_sub) == 0:
        print(f"  ({country}, ecoregion {ecoregion}): no controls at any fallback level, "
              f"skipping {len(treat_sub)} treatment cells")
        continue

    # NN search within this stratum's control pool
    n_neighbors = min(N_NEIGHBORS, len(control_sub))
    nn = NearestNeighbors(n_neighbors=n_neighbors, metric="euclidean")
    nn.fit(control_sub[["propensity_score"]].values)

    distances, indices = nn.kneighbors(treat_sub[["propensity_score"]].values)

    for i, treat_row in enumerate(treat_sub.itertuples()):
        for rank, (dist, j) in enumerate(zip(distances[i], indices[i]), start=1):
            if dist <= CALIPER:
                control_row = control_sub.iloc[j]
                matches.append({
                    "treat_cell_id": treat_row.cell_ID,
                    "control_cell_id": control_row["cell_ID"],
                    "treat_score": treat_row.propensity_score,
                    "control_score": control_row["propensity_score"],
                    "ps_distance": float(dist),
                    "match_rank": rank,
                    "match_country": country,
                    "match_ecoregion": ecoregion,
                    "match_fallback": fallback,
                })

match_df = pd.DataFrame(matches).sort_values("treat_cell_id").reset_index(drop=True)

print(f"\nResults:")
print(f"  Total matched pairs: {len(match_df)}")
print(f"  Unique treatment cells matched: {match_df['treat_cell_id'].nunique()}")
print(f"  Unique control cells used: {match_df['control_cell_id'].nunique()}")

unmatched_treat = set(treat_df["cell_ID"]) - set(match_df["treat_cell_id"])
print(f"  Treatment cells with no match: {len(unmatched_treat)}")

match_coverage = match_df["treat_cell_id"].nunique() / len(treat_df)
print(f"  Match coverage: {match_coverage:.1%}")

if len(match_df) > 0:
    avg_matches = match_df.groupby("treat_cell_id").size().mean()
    print(f"  Avg matches per matched treatment cell: {avg_matches:.2f}")

    control_reuse = match_df.groupby("control_cell_id").size()
    print(f"  Control reuse: min={control_reuse.min()}, "
          f"max={control_reuse.max()}, "
          f"mean={control_reuse.mean():.1f}")

print(f"\nFirst 10 matches:")
print(match_df.head(10) if len(match_df) > 0 else "(no matches)")

In [ ]:
# Filter grid cells to only valid matches

valid_ids = pd.concat([match_df["treat_cell_id"], match_df["control_cell_id"]]).unique()
valid_ids = ee.List(valid_ids.astype(int).tolist())
matched_grids = grid_fc.filter(ee.Filter.inList("cell_ID", valid_ids))

In [ ]:
# Save matched_grids and propensity match pairs to parquet
matched_grids_gdf = geemap.ee_to_gdf(matched_grids)
matched_grids_gdf.to_parquet(f"data/matched_grids_{PA_ID}.parquet")
match_df.to_parquet(f"data/match_table_{PA_ID}.parquet", index=False)

## Diagnostics

In [ ]:
# === Diagnostic 1: Data quality ===
# Verifies inputs are valid and identifies any silent data losses through the pipeline.

print(f"PA {PA_ID} — Data Quality Check")
print("=" * 60)

# Cell counts at each stage
n_treat_in = len(treatment_cells)
n_control_in = len(control_cells)
n_in_total = n_treat_in + n_control_in
n_after_covariates = len(cells_df)  # after reduceRegions + categorical join + getInfo
n_lost = n_in_total - n_after_covariates

print(f"\nCells loaded from parquet:    {n_in_total:>6} ({n_treat_in} treatment, {n_control_in} control)")
print(f"Cells after extraction:       {n_after_covariates:>6}")
print(f"Cells dropped:                {n_lost:>6} ({n_lost/n_in_total:.1%})")

if n_lost > 0:
    pct_lost = n_lost / n_in_total
    if pct_lost > 0.05:
        print(f"  ⚠ {pct_lost:.1%} loss is unusually high — investigate which cells dropped")

# Missingness on key columns
print(f"\nMissingness:")
for col in ["country", "ecoregion", "biome", "elevation", "slope", "treecover2000", "travel_time", "log_pop_density"]:
    n_missing = cells_df[col].isna().sum()
    if n_missing > 0:
        print(f"  {col:20s}: {n_missing} cells ({n_missing/len(cells_df):.1%})")
if not cells_df[["country", "ecoregion", "biome", "elevation", "slope", "treecover2000", "travel_time", "log_pop_density"]].isna().any().any():
    print("  (no missing values)")

# Geographic context
print(f"\nGeographic context:")
print(f"  Countries:  {sorted(cells_df['country'].dropna().unique())}")
print(f"  Ecoregions: {sorted(cells_df['ecoregion'].dropna().unique().astype(int))}")
print(f"  Biomes:     {sorted(cells_df['biome'].dropna().unique().astype(int))}")

# Cross-tabulate treatment cells by stratum to spot cross-border PAs
print(f"\nTreatment cells by (country, ecoregion, biome):")
print(cells_df[cells_df["protected"] == 1].groupby(["country", "ecoregion", "biome"]).size().to_string())

In [ ]:
# === Diagnostic 2: Stratum coverage ===
# Checks whether each treatment stratum has enough controls available for matching.

print(f"PA {PA_ID} — Stratum Coverage")
print("=" * 60)

treat_df = cells_df[cells_df["protected"] == 1]
control_df = cells_df[cells_df["protected"] == 0]

stratum_summary = []
for (country, ecoregion), treat_sub in treat_df.groupby(["country", "ecoregion"]):
    biome = treat_sub["biome"].iloc[0]
    
    n_same_eco = len(control_df[(control_df["country"] == country) & (control_df["ecoregion"] == ecoregion)])
    n_same_biome = len(control_df[(control_df["country"] == country) & (control_df["biome"] == biome)])
    n_same_country = len(control_df[control_df["country"] == country])
    
    stratum_summary.append({
        "country": country,
        "ecoregion": int(ecoregion),
        "biome": int(biome),
        "n_treatment": len(treat_sub),
        "n_same_ecoregion": n_same_eco,
        "n_same_biome": n_same_biome,
        "n_same_country": n_same_country,
        "fallback_likely": "ecoregion" if n_same_eco >= 4 else ("biome" if n_same_biome >= 4 else "INSUFFICIENT"),
    })

stratum_df = pd.DataFrame(stratum_summary)
print(stratum_df.to_string(index=False))

# Flags
insufficient = stratum_df[stratum_df["fallback_likely"] == "INSUFFICIENT"]
if len(insufficient) > 0:
    n_unmatchable = insufficient["n_treatment"].sum()
    print(f"\n⚠ {len(insufficient)} stratum/strata have insufficient controls — {n_unmatchable} treatment cells likely unmatchable")

needs_fallback = stratum_df[stratum_df["fallback_likely"] == "biome"]
if len(needs_fallback) > 0:
    n_fallback = needs_fallback["n_treatment"].sum()
    print(f"⚠ {len(needs_fallback)} stratum/strata will fall back to biome — {n_fallback} treatment cells affected")

In [ ]:
# === Diagnostic 3: Propensity score & matching quality ===
# Inspects the propensity score distribution and the resulting matches.

print(f"PA {PA_ID} — Propensity Scores & Matching")
print("=" * 60)

t_scores = cells_df.loc[cells_df["protected"] == 1, "propensity_score"]
c_scores = cells_df.loc[cells_df["protected"] == 0, "propensity_score"]

print(f"\nPropensity score distribution:")
print(f"                    treatment    control     diff")
print(f"  mean:              {t_scores.mean():>8.4f}   {c_scores.mean():>8.4f}   {t_scores.mean() - c_scores.mean():>+8.4f}")
print(f"  std:               {t_scores.std():>8.4f}   {c_scores.std():>8.4f}")
print(f"  min:               {t_scores.min():>8.4f}   {c_scores.min():>8.4f}")
print(f"  25%:               {t_scores.quantile(0.25):>8.4f}   {c_scores.quantile(0.25):>8.4f}")
print(f"  50%:               {t_scores.median():>8.4f}   {c_scores.median():>8.4f}")
print(f"  75%:               {t_scores.quantile(0.75):>8.4f}   {c_scores.quantile(0.75):>8.4f}")
print(f"  max:               {t_scores.max():>8.4f}   {c_scores.max():>8.4f}")

# Distribution sanity checks
if t_scores.mean() <= c_scores.mean():
    print(f"\n⚠ Treatment mean propensity ≤ control mean — model may be inverted for this PA")

t_range = t_scores.max() - t_scores.min()
overlap_low = max(t_scores.min(), c_scores.min())
overlap_high = min(t_scores.max(), c_scores.max())
overlap_pct = max(0, (overlap_high - overlap_low) / t_range) if t_range > 0 else 0
print(f"\nOverlap region (where matching can find candidates): [{overlap_low:.3f}, {overlap_high:.3f}]")
print(f"Treatment cells in overlap region: {((t_scores >= overlap_low) & (t_scores <= overlap_high)).sum()}/{len(t_scores)} ({((t_scores >= overlap_low) & (t_scores <= overlap_high)).mean():.1%})")

# Matching outcomes
print(f"\nMatching results:")
n_unique_treat_matched = match_df["treat_cell_id"].nunique()
n_unique_control_used = match_df["control_cell_id"].nunique()
n_treatment_total = len(treat_df)
n_control_total = len(control_df)

print(f"  Total matched pairs:           {len(match_df)}")
print(f"  Treatment match coverage:      {n_unique_treat_matched}/{n_treatment_total} ({n_unique_treat_matched/n_treatment_total:.1%})")
print(f"  Unique controls used:          {n_unique_control_used}/{n_control_total} ({n_unique_control_used/n_control_total:.1%})")
print(f"  Mean matches per treatment:    {len(match_df) / n_unique_treat_matched:.2f}")

control_reuse = match_df.groupby("control_cell_id").size()
print(f"  Control reuse:                 mean={control_reuse.mean():.1f}, max={control_reuse.max()}, median={control_reuse.median():.0f}")

print(f"\n  Propensity distance stats:")
print(f"    mean:  {match_df['ps_distance'].mean():.4f}")
print(f"    max:   {match_df['ps_distance'].max():.4f}")
print(f"    >0.05: {(match_df['ps_distance'] > 0.05).sum()} pairs ({(match_df['ps_distance'] > 0.05).mean():.1%})")

print(f"\n  Fallback usage:")
fallback_counts = match_df["match_fallback"].fillna("ecoregion").value_counts()
for label, n in fallback_counts.items():
    print(f"    {label}: {n} pairs ({n/len(match_df):.1%})")

# Red flags
if n_unique_treat_matched / n_treatment_total < 0.8:
    print(f"\n⚠ Match coverage <80% — significant fraction of PA unanalyzable")
if control_reuse.mean() > 10:
    print(f"⚠ Mean control reuse >10 — control pool may be too small for reliable matching")
if (match_df["match_fallback"] == "biome").mean() > 0.5:
    print(f"⚠ >50% of matches used biome fallback — ecoregion constraint may be too tight")

In [ ]:
# === Diagnostic 4: Covariate extrapolation ===
# Checks whether PA covariates fall within the range of the training data.
# Predictions outside training range are extrapolation and should be treated with caution.

print(f"PA {PA_ID} — Covariate Extrapolation Check")
print("=" * 60)

training_data = pd.read_parquet("data/samples_thinned.parquet")

if training_data is not None:
    print(f"\n{'covariate':<20} {'PA range':<25} {'Training p1-p99':<25} {'Out of range':<15}")
    print("-" * 85)
    
    any_extrapolation = False
    for col in ["elevation", "slope", "treecover2000", "travel_time", "log_pop_density"]:
        pa_vals = cells_df[col]
        train_vals = training_data[col]
        
        # Use 1st-99th percentile of training data as the "valid" range
        # (more robust than absolute min/max which is sensitive to outliers)
        train_p1 = train_vals.quantile(0.01)
        train_p99 = train_vals.quantile(0.99)
        
        n_low = (pa_vals < train_p1).sum()
        n_high = (pa_vals > train_p99).sum()
        n_out = n_low + n_high
        
        pa_range_str = f"[{pa_vals.min():.1f}, {pa_vals.max():.1f}]"
        train_range_str = f"[{train_p1:.1f}, {train_p99:.1f}]"
        out_str = f"{n_out} ({n_out/len(pa_vals):.0%})"
        
        if n_out > 0:
            any_extrapolation = True
            out_str += " ⚠"
        
        print(f"{col:<20} {pa_range_str:<25} {train_range_str:<25} {out_str:<15}")
    
    if any_extrapolation:
        print(f"\n⚠ Some PA cells have covariate values outside the training distribution.")
        print(f"  Propensity scores for these cells are extrapolations and may be unreliable.")

In [ ]:
# === Diagnostic 5: Covariate covariate balance check ===

covariates_list = ["elevation", "slope", "treecover2000", "travel_time", "log_pop_density"]

# Build matched treatment and control DataFrames
# Each row of match_df becomes one row in each: treat row has treatment covariates,
# control row has control covariates. Controls appear multiple times (reuse).
matched_treat = match_df.merge(
    cells_df[["cell_ID"] + covariates_list],
    left_on="treat_cell_id",
    right_on="cell_ID",
).drop(columns="cell_ID")

matched_control = match_df.merge(
    cells_df[["cell_ID"] + covariates_list],
    left_on="control_cell_id",
    right_on="cell_ID",
).drop(columns="cell_ID")

def compute_smd(t_vals, c_vals):
    """Standardized mean difference. Returns 0 if both groups have zero variance."""
    mean_t, mean_c = t_vals.mean(), c_vals.mean()
    var_t, var_c = t_vals.var(), c_vals.var()
    pooled_sd = np.sqrt((var_t + var_c) / 2)
    if pooled_sd == 0:
        return 0.0
    return (mean_t - mean_c) / pooled_sd

print("Balance check (SMD = (mean_T - mean_C) / pooled_SD)")
print("Threshold: |SMD| < 0.1 indicates good balance")
print("=" * 80)
print(f"{'covariate':<20s} {'before':>10s} {'after':>10s} {'verdict':>20s}")
print("-" * 80)

# Before-matching: full treatment vs control pools (no reuse, no matches yet)
unmatched_treat = cells_df[cells_df["protected"] == 1][covariates_list]
unmatched_control = cells_df[cells_df["protected"] == 0][covariates_list]

for col in covariates_list:
    before = compute_smd(unmatched_treat[col], unmatched_control[col])
    after = compute_smd(matched_treat[col], matched_control[col])

    if abs(after) < 0.1:
        verdict = "balanced"
    elif abs(after) < 0.25:
        verdict = "borderline"
    else:
        verdict = "IMBALANCED"

    print(f"{col:<20s} {before:>+10.3f} {after:>+10.3f} {verdict:>20s}")

## Visualization

In [ ]:
Map = geemap.Map()
Map.add_basemap("CartoDB.Positron")
ecoRegions = ee.FeatureCollection(BIOME_ASSET_ID)

color_updates = [
    {"ECO_ID": 204, "COLOR": '#B3493B'},
    {"ECO_ID": 245, "COLOR": '#267400'},
    {"ECO_ID": 259, "COLOR": '#004600'},
    {"ECO_ID": 286, "COLOR": '#82F178'},
    {"ECO_ID": 316, "COLOR": '#E600AA'},
    {"ECO_ID": 453, "COLOR": '#5AA500'},
    {"ECO_ID": 317, "COLOR": '#FDA87F'},
    {"ECO_ID": 763, "COLOR": '#A93800'},
]

def add_style_property(feature):
    color = feature.get('COLOR')
    return feature.set('style', {'color': color, 'width': 0})
ecoRegions = ecoRegions.map(add_style_property)

for update in color_updates:
    layer = ecoRegions.filter(ee.Filter.eq('ECO_ID', update['ECO_ID'])).map(
        lambda f: f.set({'COLOR': update['COLOR'], 'style': {'color': update['COLOR'], 'width': 0}})
    )
    ecoRegions = ecoRegions.filter(ee.Filter.neq('ECO_ID', update['ECO_ID'])).merge(layer)

ecoRegions = ecoRegions.style(**{'styleProperty': 'style'})

land_mask = (
    ee.Image(HGFC_ASSET_ID)
    .select("datamask")
    .eq(1)  # 1 = land, 2 = permanent water/ocean, 0 = no data
)

Map.addLayer(ecoRegions.updateMask(land_mask), {}, 'Ecoregions')
Map.addLayer(grid_fc, {"color": "black"}, "Candidate Cells")
Map.addLayer(matched_grids, {"color": "yellow"}, "Matched Cells")
Map.centerObject(grid_fc)
Map
